<a href="https://colab.research.google.com/github/krishgupta1843-star/project-beach/blob/main/CreditRiskML_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import joblib


In [ ]:
df = pd.read_csv('credit_risk_dataset.csv')
print(f'✅ Dataset loaded successfully!')
print(f'   Rows    : {df.shape[0]}')
print(f'   Columns : {df.shape[1]}')
print(f'\nColumn Names: {df.columns.tolist()}')
print(f'\nData Types:\n{df.dtypes}')
print('\n── First 5 rows ──')
display(df.head())
print('\n── Last 5 rows ──')
display(df.tail())

In [ ]:
df.info()
display(df.describe())
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing[missing > 0])
print(f'\n── Duplicate Rows: {df.duplicated().sum()} ──')
print('\n── Unique Values per Column ──')
print(df.nunique())
print('\n── Target Column (Default) Distribution ──')
print(df['Default'].value_counts())
print(f'\nClass 0 (No Default) : {(df["Default"]==0).sum()} ({(df["Default"]==0).mean()*100:.1f}%)')
print(f'Class 1 (Default)    : {(df["Default"]==1).sum()} ({(df["Default"]==1).mean()*100:.1f}%)')

In [ ]:
df_clean = df.copy()
missing_pct = df_clean.isnull().sum() / len(df_clean) * 100
cols_to_drop = missing_pct[missing_pct > 20].index.tolist()
if cols_to_drop:
    df_clean.drop(columns=cols_to_drop, inplace=True)
    print(f'  Dropped columns: {cols_to_drop}')
else:
    print('nothing dropped')
num_cols = df_clean.select_dtypes(include='number').columns
cat_cols = df_clean.select_dtypes(include='object').columns
for col in num_cols:
    if df_clean[col].isnull().any():
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f'{col}: filled with median = {median_val}')

for col in cat_cols:
    if df_clean[col].isnull().any():
        mode_val = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(mode_val)
        print(f' {col}: filled with mode = {mode_val}')
print('\n── Rule 3: Remove duplicate rows ──')
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'  Duplicates removed: {before - len(df_clean)}')
print('\n── Rule 4: Strip spaces from categorical values ──')
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print('\n── Rule 5: Standardise column names ──')
df_clean.columns = [c.strip().lower().replace(' ', '_') for c in df_clean.columns]
print(f'Column names: {df_clean.columns.tolist()}')

print(f'\nCleaning complete! Shape: {df_clean.shape}')
print(f'   Remaining nulls: {df_clean.isnull().sum().sum()}')

In [ ]:

num_df   = df_clean.select_dtypes(include='number')
corr_mat = num_df.corr().abs()
upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
redundant = [col for col in upper.columns if any(upper[col] > 0.95)]

print('\n── Highly Correlated Pairs (|r| > 0.95) ──')
if redundant:
    for col in redundant:
        partners = upper.index[upper[col] > 0.95].tolist()
        print(f'  ✂  Dropping: {col}  (correlated with {partners})')
    df_clean.drop(columns=redundant, inplace=True)
else:
    print('No highly correlated pairs found. All columns retained.')

print(f'\nFinal shape after redundancy check: {df_clean.shape}')
print(f' Columns: {df_clean.columns.tolist()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Target Variable — Default Distribution', fontsize=14, fontweight='bold')
sns.countplot(data=df_clean, x='default', palette=['steelblue','tomato'], ax=axes[0])
axes[0].set_title('Count of Default vs No Default')
axes[0].set_xticklabels(['No Default (0)', 'Default (1)'])
df_clean['default'].value_counts().plot.pie(
    ax=axes[1], autopct='%1.1f%%',
    labels=['No Default', 'Default'],
    colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_ylabel('')
axes[1].set_title('Percentage Split')

plt.tight_layout()
plt.show()


In [ ]:
num_features = ['age', 'income', 'loan_amount', 'credit_score', 'employment_years']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribution of Numerical Features', fontsize=14, fontweight='bold')
axes = axes.flatten()
for i, col in enumerate(num_features):
    sns.histplot(df_clean[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(col.replace('_', ' ').title())

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Boxplots — Outlier Detection by Default Status', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.boxplot(data=df_clean, x='default', y=col,
                palette=['steelblue', 'tomato'], ax=axes[i])
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_xticklabels(['No Default', 'Default'])

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
cat_features = ['education_level', 'housing_status']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Categorical Features vs Default', fontsize=14, fontweight='bold')
for ax, col in zip(axes, cat_features):
    sns.countplot(data=df_clean, x=col, hue='default',
                  palette=['steelblue', 'tomato'], ax=ax)
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(title='Default', labels=['No', 'Yes'])
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 6))
corr = df_clean.select_dtypes(include='number').corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', mask=mask,
            cmap='coolwarm', linewidths=0.5, annot_kws={'size': 9})
plt.title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
TARGET = 'default'
X = df_clean.drop(columns=[TARGET])

y = df_clean[TARGET]
print(f'   X shape : {X.shape}  (features)')
print(f'   y shape : {y.shape}  (target)')
print(f'\n   Feature columns : {X.columns.tolist()}')
print(f'   Target column   : {TARGET}')
print(f'   Target classes  : {y.unique()} (0=No Default, 1=Default)')

In [ ]:
X_encoded = pd.get_dummies(X, columns=['education_level', 'housing_status'],
                            drop_first=True)
bool_cols = X_encoded.select_dtypes(include='bool').columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)
print(f'   Shape before encoding : {X.shape}')
print(f'   Shape after encoding  : {X_encoded.shape}')
print(f'\n   All columns after encoding:')
print(X_encoded.columns.tolist())
display(X_encoded.head(3))

In [ ]:
num_cols_to_scale = ['age', 'income', 'loan_amount', 'credit_score', 'employment_years']
scaler = StandardScaler()
X_encoded[num_cols_to_scale] = scaler.fit_transform(X_encoded[num_cols_to_scale])
print(f'   Columns scaled: {num_cols_to_scale}')
display(X_encoded[num_cols_to_scale].head(3).round(3))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'   Training samples : {X_train.shape[0]}  (80%)')
print(f'   Testing  samples : {X_test.shape[0]}   (20%)')
print(y_train.value_counts())
print(y_test.value_counts())

In [ ]:
models = {
    'Logistic Regression'    : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'          : DecisionTreeClassifier(random_state=42),
    'Random Forest'          : RandomForestClassifier(n_estimators=100, random_state=42),
    'K-Nearest Neighbors'    : KNeighborsClassifier(n_neighbors=5),
    'Support Vector Machine' : SVC(kernel='rbf', probability=True, random_state=42),
    'Naive Bayes'            : GaussianNB()
}
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f'Trained : {name}')

In [ ]:

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x      = np.arange(len(results_df))
width  = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
colors  = ['steelblue', 'seagreen', 'tomato', 'orange']

for i, metric in enumerate(metrics):
    ax.bar(x + i * width, results_df[metric], width, label=metric, color=colors[i])

ax.set_xlabel('Models')
ax.set_ylabel('Score (%)')
ax.set_title('Model Comparison — All Metrics', fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df['Model'], rotation=20, ha='right')
ax.legend()
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

In [ ]:
best_model  = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test)
cm          = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title(f'Confusion Matrix — {best_model_name}', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print(f'\n── Classification Report — {best_model_name} ──')
print(classification_report(y_test, y_pred_best,
      target_names=['No Default', 'Default']))

In [ ]:
y_pred_final = best_model.predict(X_test)
prediction_df = pd.DataFrame({
    'Actual'    : y_test.values[:10],
    'Predicted' : y_pred_final[:10]
})
prediction_df['Actual Label']    = prediction_df['Actual'].map({0: 'No Default', 1: 'Default'})
prediction_df['Predicted Label'] = prediction_df['Predicted'].map({0: 'No Default', 1: 'Default'})
prediction_df['Correct?']        = prediction_df['Actual'] == prediction_df['Predicted']
prediction_df['Correct?']        = prediction_df['Correct?'].map({True: '✅', False: '❌'})

print(f'── First 10 Predictions using {best_model_name} ──')
display(prediction_df)

correct = (y_pred_final == y_test.values).sum()
print(f'\nCorrect predictions on full test set: {correct} / {len(y_test)}')

In [ ]:
rf_model     = trained_models['Random Forest']
importances  = rf_model.feature_importances_
feat_imp_df  = pd.DataFrame({
    'Feature'   : X_encoded.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('── Top 5 Most Important Features ──')
print(feat_imp_df.head(5).to_string(index=False))
print('\n💡 Insight: Age, Credit Score, and Income are the strongest predictors of loan default.')

In [ ]:
model_filename = 'best_credit_risk_model.pkl'
joblib.dump(best_model, model_filename)
print(f'Model saved as: {model_filename}')
scaler_filename = 'feature_scaler.pkl'
joblib.dump(scaler, scaler_filename)
print(f'Scaler saved as: {scaler_filename}')
print("""
  import joblib
  loaded_model  = joblib.load('best_credit_risk_model.pkl')
  loaded_scaler = joblib.load('feature_scaler.pkl')
  prediction    = loaded_model.predict(new_data)
""")

In [ ]:
best_row = results_df.iloc[0]

print('═' * 65)
print('         💳 CREDIT RISK PREDICTION — PROJECT SUMMARY')
print('═' * 65)
print(f'  Dataset        : credit_risk_dataset.csv')
print(f'  Problem Type   : Binary Classification')
print(f'  Target Column  : default (0 = No Default, 1 = Default)')
print(f'  Dataset Size   : 1000 rows × 8 columns')
print(f'  Models Trained : 6 ML models')
print(f'  Best Model     : {best_row["Model"]}')
print(f'  Accuracy       : {best_row["Accuracy"]}%')
print(f'  F1 Score       : {best_row["F1 Score"]}%')
print('─' * 65)
print('  TOP FEATURES   : Age, Credit Score, Income')
print('─' * 65)
print('  ADVANTAGES:')
print('    • Fast to train and predict')
print('    • Interpretable feature importance')
print('    • No deep learning required')
print('  LIMITATIONS:')
print('    • Dataset is imbalanced (86% vs 14%)')
print('    • Small dataset (only 1000 rows)')
print('  FUTURE IMPROVEMENTS:')
print('    • Apply SMOTE to handle class imbalance')
print('    • Hyperparameter tuning with GridSearchCV')
print('    • Use larger real-world banking datasets')
print('    • Deploy as a Flask/FastAPI web application')
print('═' * 65)
print('  ✅ PROJECT COMPLETE — INTERNSHIP SUBMISSION READY')
print('═' * 65)